In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score)
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, chi2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("Test.csv")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
print(df_train.head())
print("***************************************************")
print(df_train.info())
print("***************************************************")
print(df_train.describe())

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities LotConfig LandSlope Neighborhood Condition1  \
0         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
1         Lvl    AllPub       FR2       Gtl      Veenker      Feedr   
2         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
3         Lvl    AllPub    Corner       Gtl      Crawfor       Norm   
4         Lvl    AllPub       FR2       Gtl      NoRidge       Norm   

  Condition2 BldgType HouseStyle  OverallQual  OverallCond  YearBuilt  \
0       Norm     1Fam     2Story            7          

In [4]:
print("Is Null:")
fields_with_null_values = df_train.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_train.dtypes[fields_with_null_values.index]
})

print(null_info)

Is Null:
              Missing Values Data Type
LotFrontage              259   float64
Alley                   1369       str
MasVnrType               872       str
MasVnrArea                 8   float64
BsmtQual                  37       str
BsmtCond                  37       str
BsmtExposure              38       str
BsmtFinType1              37       str
BsmtFinType2              38       str
Electrical                 1       str
FireplaceQu              690       str
GarageType                81       str
GarageYrBlt               81   float64
GarageFinish              81       str
GarageQual                81       str
GarageCond                81       str
PoolQC                  1453       str
Fence                   1179       str
MiscFeature             1406       str


In [5]:
df_train["LotFrontage"] = (
    df_train.groupby("Neighborhood")["LotFrontage"]
      .transform(lambda x: x.fillna(x.median()))
)

df_train["Electrical"] = df_train["Electrical"].fillna(
    df_train["Electrical"].mode()[0]
)

df_train["Alley"] = df_train["Alley"].fillna("None")
df_train["MasVnrType"] = df_train["MasVnrType"].fillna("None")
df_train["MasVnrArea"] = df_train["MasVnrArea"].fillna(0)
df_train["BsmtQual"] = df_train["BsmtQual"].fillna("None")
df_train["BsmtCond"] = df_train["BsmtCond"].fillna("None")
df_train["BsmtExposure"] = df_train["BsmtExposure"].fillna("None")
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].fillna("None")
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].fillna("None")
df_train["FireplaceQu"] = df_train["FireplaceQu"].fillna("None")
df_train["GarageType"] = df_train["GarageType"].fillna("None")
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].fillna(0)
df_train["GarageFinish"] = df_train["GarageFinish"].fillna("None")
df_train["GarageQual"] = df_train["GarageQual"].fillna("None")
df_train["GarageCond"] = df_train["GarageCond"].fillna("None")
df_train["PoolQC"] = df_train["PoolQC"].fillna("None")
df_train["Fence"] = df_train["Fence"].fillna("None")
df_train["MiscFeature"] = df_train["MiscFeature"].fillna("None")

In [6]:
print("Is Null:")
fields_with_null_values = df_test.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_test.dtypes[fields_with_null_values.index]
})

print(null_info)

Is Null:
              Missing Values Data Type
MSZoning                   4       str
LotFrontage              227   float64
Alley                   1352       str
Utilities                  2       str
Exterior1st                1       str
Exterior2nd                1       str
MasVnrType               894       str
MasVnrArea                15   float64
BsmtQual                  44       str
BsmtCond                  45       str
BsmtExposure              44       str
BsmtFinType1              42       str
BsmtFinSF1                 1   float64
BsmtFinType2              42       str
BsmtFinSF2                 1   float64
BsmtUnfSF                  1   float64
TotalBsmtSF                1   float64
BsmtFullBath               2   float64
BsmtHalfBath               2   float64
KitchenQual                1       str
Functional                 2       str
FireplaceQu              730       str
GarageType                76       str
GarageYrBlt               78   float64
GarageFinish    

In [7]:
neighborhood_avg = [
  "LotFrontage"
]

str_category = [
  "MSZoning",
  "Utilities",
  "Exterior1st",
  "Exterior2nd",
  "SaleType"
]

na_to_none = [
  "Alley",
  "MasVnrType",
  "BsmtQual",
  "BsmtCond",
  "BsmtExposure",
  "BsmtFinType1",
  "BsmtFinType2",
  "KitchenQual",
  "Functional",
  "FireplaceQu",
  "GarageType",
  "GarageFinish",
  "GarageQual",
  "GarageCond",
  "PoolQC",
  "Fence",
  "MiscFeature"
]

na_to_0 = [
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtUnfSF",
  "TotalBsmtSF",
  "BsmtFullBath",
  "BsmtHalfBath",
  "MasVnrArea",
  "GarageYrBlt",
  "GarageCars",
  "GarageArea"
]

for item in neighborhood_avg:
  df_test[item] = (
    df_test.groupby("Neighborhood")[item]
    .transform(lambda x: x.fillna(x.median()))
  )

for item in str_category:
  df_test[item] = df_test[item].fillna(
      df_test[item].mode()[0]
  )

for item in na_to_none:
  df_test[item] = df_test[item].fillna("None")

for item in na_to_0:
  df_test[item] = df_test[item].fillna(0)

In [8]:
# Creating Classification Dataset
classification_df_train = df_train.copy()

# 0 = Low, 1 = Medium, 2 = High
classification_df_train["PriceCategory"] = pd.qcut(
    classification_df_train["SalePrice"],
    q=3,
    labels=[0, 1, 2]
)

classification_df_train.drop(
    columns=["SalePrice"],
    inplace=True
)
classification_df_train["PriceCategory"].value_counts()


PriceCategory
1    490
0    487
2    483
Name: count, dtype: int64

Nominal Catagories:
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"

Y/N:
"CentralAir"

Ordinal 1-10:
"OverallQual",
"OverallCond"

Ordinal Po-Ex(5):
"ExterQual",
"ExterCond",
"HeatingQC",
"KitchenQual"

Ordinal Na-Ex(6):
"BsmtQual",
"BsmtCond",
"FireplaceQu",
"GarageQual",
"GarageCond"

Ordinal Na-Ex(5):
"PoolQC"

Ordinal Misc:
"BsmtExposure",
"BsmtFinType1",
"BsmtFinType2",
"Functional",
"GarageFinish",
"PavedDrive"??



In [9]:
#Ordinal Maps
datasets = [
  df_train,
  classification_df_train,
  df_test
]

for dataset in datasets:
  # Yes/No Map
  quality_map = {
    "N": 0,
    "Y": 1
  }

  dataset["CentralAir"] = dataset["CentralAir"].map(quality_map)

  # Poor - Excellent 5 Rankings Map
  quality_map = {
    "Po": 0,
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_po_ex_5 = [
  "ExterQual",
  "ExterCond",
  "HeatingQC",
  "KitchenQual"
  ]

  for category in ordinal_po_ex_5:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 6 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
  }

  ordinal_na_ex_6 = [
  "BsmtQual",
  "BsmtCond",
  "FireplaceQu",
  "GarageQual",
  "GarageCond"
  ]

  for category in ordinal_na_ex_6:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 5 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_na_ex_5 = [
    "PoolQC"
  ]

  for category in ordinal_na_ex_5:
    dataset[category]=dataset[category].map(quality_map)


  # Misc. Ordinal Rankings Map

  # "BsmtExposure"
  quality_map ={
    "NA": 0,
    "None": 0,
    "No": 0,
    "Mn":	2,
    "Av":	3,
    "Gd":	4
  }

  dataset["BsmtExposure"]=dataset["BsmtExposure"].map(quality_map)

  # "BsmtFinType1"
  # "BsmtFinType2"
  quality_map = {
    "NA": 0,
    "None": 0,
    "No": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
  }

  dataset["BsmtFinType1"]=dataset["BsmtFinType1"].map(quality_map)
  dataset["BsmtFinType2"]=dataset["BsmtFinType2"].map(quality_map)

  # "Functional"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
  }
  dataset["Functional"]=dataset["Functional"].map(quality_map)

  # "GarageFinish"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
  }

  dataset["GarageFinish"]=dataset["GarageFinish"].map(quality_map)


  # "PavedDrive"
  quality_map ={
    "N": 0,
    "NA": 0,
    "None": 0,
    "P" : 1,
    "Y": 2
  }

  dataset["PavedDrive"]=dataset["PavedDrive"].map(quality_map)

In [10]:
# Regression dataset targeting sale price
X = df_train.drop(columns=["SalePrice"])
y=df_train["SalePrice"]



# Classification dataset targeting price category 
y_class = classification_df_train["PriceCategory"]
X_class = classification_df_train.drop(columns=["PriceCategory"])

X_test = df_test

In [11]:
#Nominal Maps
nominal_columns = [
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",
        OneHotEncoder(handle_unknown="ignore", drop="first"),
        nominal_columns)
    ],
    remainder="passthrough"
)

X = preprocessor.fit_transform(X)
X_class = preprocessor.transform(X_class)
X_test = preprocessor.transform(X_test)

c:\code\d789_task2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [12]:
# Regression Split
X_train, X_val, y_train, y_val = train_test_split(
  X,
  y,
  test_size=0.2,
  random_state=42
)

# Classification Split
X_class_train, X_class_val, y_class_train, y_class_val = train_test_split(
  X_class,
  y_class,
  test_size=0.2,
  random_state=42
)


In [13]:
# Chi-Square Feature Selection - Classification Dataset - (Running before scaling to avoid negatives)
selector = SelectKBest(
  score_func=chi2,
  k=20
)

X_class_train_chi2 = selector.fit_transform(X_class_train, y_class_train)
X_class_val_chi2 = selector.transform(X_class_val)

In [14]:

# Scale regression dataset
reg_scaler = StandardScaler()
X_train = reg_scaler.fit_transform(X_train)
X_val = reg_scaler.transform(X_val)

# Scale classification dataset
class_scaler = StandardScaler()
X_class_train = class_scaler.fit_transform(X_class_train)
X_class_val = class_scaler.transform(X_class_val)

# Scale Chi-Square Classification Dataset
chi2_scaler = StandardScaler()
X_class_train_chi2 = chi2_scaler.fit_transform(X_class_train_chi2)
X_class_val_chi2 = chi2_scaler.transform(X_class_val_chi2)


# Fix classification dataset label type
y_class_train = y_class_train.astype("int32")
y_class_val = y_class_val.astype("int32")

In [15]:
# PCA Feature Selection - Regression DataSet
pca = PCA(n_components=0.95)

X_train_PCA = pca.fit_transform(X_train)
X_val_PCA = pca.transform(X_val)


In [16]:
# ANOVA F-Test Feature Selection - Regression & Classification Datasets
selector = SelectKBest(
  score_func=f_regression,
  k=20
)

X_train_ANOVA = selector.fit_transform(X_train, y_train)
X_val_ANOVA = selector.transform(X_val)

X_class_train_ANOVA = selector.fit_transform(X_class_train, y_class_train)
X_class_val_ANOVA = selector.transform(X_class_val)

In [17]:
# Regression Model - Linear Regression
model = LinearRegression()
results = []

# Full feature set
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model": "Linear Regression - Full Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})  


# PCA Selected Feature Set
model.fit(X_train_PCA, y_train)

y_pred = model.predict(X_val_PCA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model": "Linear Regression - PCA Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# ANOVA Selected Feature Set
model.fit(X_train_ANOVA, y_train)

y_pred = model.predict(X_val_ANOVA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model": "Linear Regression - PCA Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

results

[{'Model': 'Linear Regression - Full Feature Set',
  'MAE': 22096.547272557564,
  'RMSE': 51432.999832958274,
  'R2': 0.6551185177315693},
 {'Model': 'Linear Regression - PCA Feature Set',
  'MAE': 21709.602454151347,
  'RMSE': 34255.71922816183,
  'R2': 0.8470137686270598},
 {'Model': 'Linear Regression - PCA Feature Set',
  'MAE': 22908.69967917897,
  'RMSE': 36392.27067086933,
  'R2': 0.8273349361151224}]

In [18]:
# Classical Classification Model - Logistical Regression
model = LogisticRegression(
    max_iter=5000,
    random_state=42
)
results = []

model.fit(X_class_train, y_class_train)

y_pred = model.predict(X_class_val)

mae = mean_absolute_error(
    y_class_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_class_val,
    y_pred,
)

r2 = r2_score(
    y_class_val,
    y_pred
)

results.append({
    "Model": "Logistic Regression - Full Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# ANOVA Selected Feature Set

model.fit(X_class_train_ANOVA, y_class_train)

y_pred = model.predict(X_class_val_ANOVA)

mae = mean_absolute_error(
    y_class_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_class_val,
    y_pred,
)

r2 = r2_score(
    y_class_val,
    y_pred
)

results.append({
    "Model": "Logistic Regression - ANOVA Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# Chi-Squared Selected Feature Set
model.fit(X_class_train_chi2, y_class_train)

y_pred = model.predict(X_class_val_chi2)

mae = mean_absolute_error(
    y_class_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_class_val,
    y_pred,
)

r2 = r2_score(
    y_class_val,
    y_pred
)

results.append({
    "Model": "Logistic Regression - Chi-Squared Selected Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

results

[{'Model': 'Logistic Regression - Full Feature Set',
  'MAE': 0.20205479452054795,
  'RMSE': 0.44950505505561106,
  'R2': 0.7095360129484758},
 {'Model': 'Logistic Regression - ANOVA Feature Set',
  'MAE': 0.16095890410958905,
  'RMSE': 0.40119683960568414,
  'R2': 0.7686134340437011},
 {'Model': 'Logistic Regression - Chi-Squared Selected Feature Set',
  'MAE': 0.2226027397260274,
  'RMSE': 0.4861083931213425,
  'R2': 0.6603048287024549}]

In [19]:
results = []

# Regression ML Model - Full Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Regression - Full Feature Set",
    "loss": history.history['loss'][-1],
    "mae": history.history['mae'][-1],
    "val_loss": history.history['val_loss'][-1],
    "val_mae": history.history['val_mae'][-1]
})


# Regression ML Model - PCA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_PCA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_PCA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Regression - PCA Selected Feature Set",
    "loss": history.history['loss'][-1],
    "mae": history.history['mae'][-1],
    "val_loss": history.history['val_loss'][-1],
    "val_mae": history.history['val_mae'][-1]
})

# Regression ML Model - ANOVA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_ANOVA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Regression - ANOVA Selected Feature Set",
    "loss": history.history['loss'][-1],
    "mae": history.history['mae'][-1],
    "val_loss": history.history['val_loss'][-1],
    "val_mae": history.history['val_mae'][-1]
})

results

Epoch 1/100


c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 39144955904.0000 - mae: 181521.5938 - val_loss: 37837033472.0000 - val_mae: 181089.8281
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39134199808.0000 - mae: 181498.5312 - val_loss: 37818544128.0000 - val_mae: 181050.4219
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39097458688.0000 - mae: 181424.0625 - val_loss: 37763420160.0000 - val_mae: 180936.5312
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39003697152.0000 - mae: 181237.3594 - val_loss: 37636620288.0000 - val_mae: 180675.0156
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38815596544.0000 - mae: 180855.7031 - val_loss: 37404856320.0000 - val_mae: 180193.9375
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38490107904.0000 - mae: 180201.3906 - val_loss: 37032763392.0000 - val_mae: 179415.7812
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 37997584384.0000 - mae: 179192.9688 - val_loss: 36480835584.0000 - val_

[{'Model': 'ML Regression - Full Feature Set',
  'loss': 1200307968.0,
  'mae': 23853.74609375,
  'val_loss': 2630028544.0,
  'val_mae': 33543.921875},
 {'Model': 'ML Regression - PCA Selected Feature Set',
  'loss': 476680064.0,
  'mae': 15746.4921875,
  'val_loss': 2165189888.0,
  'val_mae': 32686.525390625},
 {'Model': 'ML Regression - ANOVA Selected Feature Set',
  'loss': 4485891072.0,
  'mae': 50605.94921875,
  'val_loss': 3026842112.0,
  'val_mae': 44533.87109375}]

In [20]:
results = []

# Classification ML Model - Full Feature Set

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Classification - Full Feature Set",
    "accuracy": history.history['accuracy'][-1],
    "loss": history.history['loss'][-1],
    "val_accuracy": history.history['val_accuracy'][-1],
    "val_loss": history.history['val_loss'][-1]
})

# Classification ML Model - ANOVA Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_ANOVA,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Classification - Full Feature Set",
    "accuracy": history.history['accuracy'][-1],
    "loss": history.history['loss'][-1],
    "val_accuracy": history.history['val_accuracy'][-1],
    "val_loss": history.history['val_loss'][-1]
})

# Classification ML Model - Chi-Squared Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_chi2.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_chi2,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Classification - Full Feature Set",
    "accuracy": history.history['accuracy'][-1],
    "loss": history.history['loss'][-1],
    "val_accuracy": history.history['val_accuracy'][-1],
    "val_loss": history.history['val_loss'][-1]
})

results

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6017 - loss: 0.8517 - val_accuracy: 0.6795 - val_loss: 0.6399
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8330 - loss: 0.4353 - val_accuracy: 0.8077 - val_loss: 0.5114
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9036 - loss: 0.3125 - val_accuracy: 0.8077 - val_loss: 0.4789
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9229 - loss: 0.2341 - val_accuracy: 0.8077 - val_loss: 0.4842
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9433 - loss: 0.1789 - val_accuracy: 0.8205 - val_loss: 0.4802
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9647 - loss: 0.1418 - val_accuracy: 0.8205 - val_loss: 0.5052
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9754 - loss: 0.1138 - val_accuracy: 0.7991 - val_loss: 0.5308
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9850 - loss: 0.0901 - val_accuracy: 0.8162 - v

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6831 - loss: 0.7256 - val_accuracy: 0.7821 - val_loss: 0.5320
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7848 - loss: 0.5050 - val_accuracy: 0.8291 - val_loss: 0.4529
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8244 - loss: 0.4552 - val_accuracy: 0.8376 - val_loss: 0.4199
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8405 - loss: 0.4219 - val_accuracy: 0.8547 - val_loss: 0.3973
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8533 - loss: 0.4013 - val_accuracy: 0.8590 - val_loss: 0.3897
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8565 - loss: 0.3805 - val_accuracy: 0.8376 - val_loss: 0.4032
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8576 - loss: 0.3713 - val_accuracy: 0.8675 - val_loss: 0.3717
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8619 - loss: 0.3521 - val_accuracy: 0.8333 - val_loss: 0.3

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5471 - loss: 0.9405 - val_accuracy: 0.6880 - val_loss: 0.7281
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7045 - loss: 0.6843 - val_accuracy: 0.7350 - val_loss: 0.6327
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7259 - loss: 0.6178 - val_accuracy: 0.7436 - val_loss: 0.6139
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7505 - loss: 0.5803 - val_accuracy: 0.7479 - val_loss: 0.6118
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7548 - loss: 0.5493 - val_accuracy: 0.7479 - val_loss: 0.6239
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7752 - loss: 0.5280 - val_accuracy: 0.7564 - val_loss: 0.6200
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7827 - loss: 0.5064 - val_accuracy: 0.7607 - val_loss: 0.6198
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7901 - loss: 0.4899 - val_accuracy: 0.7436 - val_loss: 0.6

[{'Model': 'ML Classification - Full Feature Set',
  'accuracy': 1.0,
  'loss': 0.0001397104060743004,
  'val_accuracy': 0.7948718070983887,
  'val_loss': 1.358472466468811},
 {'Model': 'ML Classification - Full Feature Set',
  'accuracy': 0.983940064907074,
  'loss': 0.05443907156586647,
  'val_accuracy': 0.8290598392486572,
  'val_loss': 0.6797061562538147},
 {'Model': 'ML Classification - Full Feature Set',
  'accuracy': 0.9935759902000427,
  'loss': 0.037301186472177505,
  'val_accuracy': 0.747863233089447,
  'val_loss': 1.2567379474639893}]